# Exploración de datos de caída libre
Este notebook es una vista exploratoria del CSV demostrativo. El pipeline oficial y reproducible se ejecuta con `python -m src.main`. Reemplaza el archivo crudo por una exportación propia de Tracker antes de redactar conclusiones.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

ROOT = Path('..').resolve() if Path('../data/raw/caida_libre_tracker.csv').exists() else Path('.').resolve()
raw = pd.read_csv(ROOT / 'data/raw/caida_libre_tracker.csv')
raw.head()

In [ ]:
raw.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(raw['t (s)'], raw['y (m)'], s=20, label='mediciones')
ax.set(xlabel='Tiempo (s)', ylabel='Posición vertical (m)', title='Exploración: y(t)')
ax.grid(True, alpha=0.3)
ax.legend();

In [ ]:
def free_fall(t, y0, v0, a):
    return y0 + v0*t + 0.5*a*t**2

t = raw['t (s)'].to_numpy()
y = raw['y (m)'].to_numpy()
params, covariance = curve_fit(free_fall, t, y)
errors = np.sqrt(np.diag(covariance))
print(f'y0 = {params[0]:.5f} ± {errors[0]:.5f} m')
print(f'v0 = {params[1]:.5f} ± {errors[1]:.5f} m/s')
print(f'a  = {params[2]:.5f} ± {errors[2]:.5f} m/s²')
plt.figure(figsize=(7,4)); plt.scatter(t, y, label='datos'); plt.plot(t, free_fall(t, *params), label='ajuste')
plt.xlabel('Tiempo (s)'); plt.ylabel('Posición (m)'); plt.grid(True, alpha=.3); plt.legend();

## Interpretación
Declara la orientación de los ejes antes de interpretar el signo de `a`. Revisa residuos, puntos de seguimiento dudosos, escala y paralaje. La matriz de covarianza calculada aquí usa el ajuste sin ponderación; para el informe utiliza el pipeline, donde se puede especificar la incertidumbre de posición y la incertidumbre relativa de escala.